In [15]:
"""
Goodreads Book ID Finder
Searches Goodreads to find book IDs for scraping
Author: Violet (Goodreads Specialist)
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
import logging

# Set up logging
logging.basicConfig(
    filename='book_id_search.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def search_goodreads_book(book_title, author=None, retries=3):
    """
    Search Goodreads for a book and return its ID and URL
    
    Parameters:
    - book_title: String, the book title to search
    - author: String (optional), author name for more specific search
    - retries: Number of times to retry if request fails
    
    Returns:
    - Dictionary with book_id, url, and found status
    """
    
    # Construct search query
    search_query = book_title
    if author:
        search_query += f" {author}"
    
    search_url = f"https://www.goodreads.com/search?q={search_query.replace(' ', '+')}"
    
    for attempt in range(retries):
        try:
            # Set headers to mimic browser
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            
            response = requests.get(search_url, headers=headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find first search result
            first_result = soup.find('a', class_='bookTitle')
            
            if first_result:
                book_url = "https://www.goodreads.com" + first_result['href']
                
                # Extract book ID from URL
                # URL format: /book/show/[ID]-title or /book/show/[ID].title
                book_id = book_url.split('/show/')[1].split('-')[0].split('.')[0]
                
                logging.info(f"Found {book_title}: ID={book_id}")
                
                return {
                    'book_title': book_title,
                    'author': author,
                    'goodreads_id': book_id,
                    'goodreads_url': book_url,
                    'found': True,
                    'search_query': search_query
                }
            else:
                logging.warning(f"No results found for: {book_title}")
                return {
                    'book_title': book_title,
                    'author': author,
                    'goodreads_id': None,
                    'goodreads_url': None,
                    'found': False,
                    'search_query': search_query
                }
                
        except Exception as e:
            if attempt < retries - 1:
                logging.warning(f"Attempt {attempt + 1} failed for {book_title}: {e}. Retrying...")
                time.sleep(5)
            else:
                logging.error(f"All attempts failed for {book_title}: {e}")
                return {
                    'book_title': book_title,
                    'author': author,
                    'goodreads_id': None,
                    'goodreads_url': None,
                    'found': False,
                    'search_query': search_query,
                    'error': str(e)
                }

import os
desktop_path = os.path.expanduser("/Users/vconklin24/Desktop/pen_america_banned_books.csv")

def find_all_book_ids(input_csv=desktop_path, output_csv='data/book_ids/goodreads_ids.csv'):
    """
    Find Goodreads IDs for all books in the dataset
    
    Parameters:
    - input_csv: Path to PEN America CSV
    - output_csv: Path to save book IDs
    """
    # Read input data
    books_df = pd.read_csv(input_csv)
    print(f"Searching for {len(books_df)} books on Goodreads...")
    
    # Initialize results list
    all_results = []
    
    # Search for each book
    for index, row in tqdm(books_df.iterrows(), total=len(books_df), desc="Searching"):
        book_title = row.get('Book Title', row.get('Title', ''))
        author = row.get('Author', None)
        
        result = search_goodreads_book(book_title, author)
        all_results.append(result)
        
        # Be respectful - wait between requests
        time.sleep(3)
        
        # Save progress every 10 books
        if (index + 1) % 10 == 0:
            temp_df = pd.DataFrame(all_results)
            temp_df.to_csv(output_csv.replace('.csv', '_temp.csv'), index=False)
    
    # Save final results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(output_csv, index=False)
    
    # Print summary
    found_count = results_df['found'].sum()
    print(f"\n=== SEARCH COMPLETE ===")
    print(f"Books found: {found_count}/{len(results_df)}")
    print(f"Books not found: {len(results_df) - found_count}")
    print(f"Results saved to: {output_csv}")
    
    # Save list of books not found
    not_found = results_df[~results_df['found']]
    if len(not_found) > 0:
        not_found.to_csv('documentation/books_not_found.csv', index=False)
        print(f"Books not found saved to: documentation/books_not_found.csv")
    
    return results_df

# Main execution
if __name__ == "__main__":
    print("\nRunning full search...")
    find_all_book_ids(
        input_csv='/Users/vconklin24/Desktop/pen_america_banned_books.csv', 
        output_csv='data/book_ids/goodreads_ids.csv'
    )
    
    # Uncomment to run full search in Week 1
    # print("\nRunning full search...")
    # find_all_book_ids()


Running full search...
Searching for 6719 books on Goodreads...


Searching:   0%|                             | 9/6719 [00:34<7:12:48,  3.87s/it]


OSError: Cannot save file into a non-existent directory: 'data/book_ids'

In [3]:
df = pd.read_csv("/Users/vconklin24/Desktop/pen_america_banned_books.csv")

In [3]:
import re

def scrape_book_details(book_id):
    url = f"https://www.goodreads.com/book/show/{book_id}"
    try:
        response = session.get(url, timeout=15)
        soup = BeautifulSoup(response.content, 'lxml' if 'lxml' in globals() else 'html.parser')
        
        title_element = soup.find('h1', {'data-testid': 'bookTitle'})
        title = title_element.text.strip() if title_element else "Unknown Title"
        
        rating_element = soup.find('div', class_='RatingStatistics__rating')
        rating = rating_element.text.strip() if rating_element else "N/A"
        
        genre_links = soup.find_all('a', href=re.compile(r'/genres/'))
        genres = list(set([g.text.strip() for g in genre_links if len(g.text.strip()) > 0]))
                pages = "N/A"
        pages_element = soup.find('p', {'data-testid': 'pagesFormat'})
        if pages_element:
            page_match = re.search(r'(\d+)', pages_element.text)
            if page_match:
                pages = page_match.group(1)

        return {
            'book_id': book_id,
            'title': title,
            'avg_rating': rating,
            'pages': pages,
            'genres': ", ".join(genres[:3]), 
            'url': url
        }
    except Exception as e:
        return {'book_id': book_id, 'error': str(e)}

In [9]:
import concurrent.futures
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import time

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
})

def scrape_book_details(book_id):
    url = f"https://www.goodreads.com/book/show/{book_id}"
    try:
        response = session.get(url, timeout=15)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Title
        title_tag = soup.find('h1', {'data-testid': 'bookTitle'})
        title = title_tag.text.strip() if title_tag else "Unknown"
        
        # Rating
        rating_tag = soup.find('div', class_='RatingStatistics__rating')
        rating = rating_tag.text.strip() if rating_tag else "N/A"
        
        # Pages (Regex fix for "Nook" issue)
        pages = "N/A"
        pages_tag = soup.find('p', {'data-testid': 'pagesFormat'})
        if pages_tag:
            page_match = re.search(r'(\d+)', pages_tag.text)
            if page_match:
                pages = page_match.group(1)
        
        # Genres
        genre_links = soup.find_all('a', href=re.compile(r'/genres/'))
        genres = list(set([g.text.strip() for g in genre_links if len(g.text.strip()) > 0]))

        return {
            'book_id': book_id,
            'title': title,
            'avg_rating': rating,
            'pages': pages,
            'genres': ", ".join(genres[:3]),
            'url': url
        }
    except Exception as e:
        return {'book_id': book_id, 'error': str(e)}

# --- EXECUTION ---
desktop = "/Users/vconklin24/Desktop"
input_path = os.path.join(desktop, "data/book_ids/goodreads_ids.csv")
output_path = os.path.join(desktop, "data/raw/full_book_details.csv")

if os.path.exists(input_path):
    ids_df = pd.read_csv(input_path)
    book_ids = ids_df[ids_df['found'] == True]['goodreads_id'].tolist()
    
    final_results = []
    print(f"Scraping {len(book_ids)} books...")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(scrape_book_details, bid): bid for bid in book_ids}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(book_ids)):
            final_results.append(future.result())

    # FIX 2: Ensuring clean CSV output
    final_df = pd.DataFrame(final_results)
    # This saves it cleanly without the "above the header" glitch
    final_df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ File saved correctly at: {output_path}")
else:
    print("❌ Could not find your goodreads_ids.csv file!")

Scraping 6031 books...


100%|███████████████████████████████████████| 6031/6031 [22:59<00:00,  4.37it/s]


✅ File saved correctly at: /Users/vconklin24/Desktop/data/raw/full_book_details.csv


In [11]:
# test scraping five books 
# 1. Load your IDs from the Desktop
ids_df = pd.read_csv(os.path.join(desktop, "data/book_ids/goodreads_ids.csv"))

# 2. Filter for the first 5 books that were actually found
test_ids = ids_df[ids_df['found'] == True]['goodreads_id'].head(5).tolist()

# 3. Scrape them using the 'session' we defined in the previous cell
print(f"Scraping details for {len(test_ids)} books to verify formatting...")
test_results = []

for bid in tqdm(test_ids):
    # We call the function we just fixed
    result = scrape_book_details(bid)
    test_results.append(result)
    time.sleep(2) # Being extra polite for the test run

# 4. Save to a specific test file
output_path_test = os.path.join(desktop, "data/raw/test_scrape.csv")
os.makedirs(os.path.dirname(output_path_test), exist_ok=True)

test_df = pd.DataFrame(test_results)
test_df.to_csv(output_path_test, index=False)

print(f"\n=== TEST COMPLETE ===")
print(f"File saved to: {output_path_test}")

# Display the results directly here in Jupyter to check the 'pages' and 'genres' columns
test_df[['title', 'avg_rating', 'pages', 'genres']]

Scraping details for 5 books to verify formatting...


100%|█████████████████████████████████████████████| 5/5 [00:15<00:00,  3.14s/it]



=== TEST COMPLETE ===
File saved to: /Users/vconklin24/Desktop/data/raw/test_scrape.csv


,title,avg_rating,pages,genres
0,Study Guide: The Perks of Being a Wallflower b...,3.98,38,Fiction
1,Study Guide: Nineteen Minutes by Jodi Picoult,4.00,80,
2,Beloved,3.98,325,"Magical Realism, Classics, School"
3,Crank by Ellen Hopkins l Summary & Study Guide,4.60,N/A,
4,Summary & Study Guide Fallout by Ellen Hopkins,4.67,31,


In [12]:
test_df

,book_id,title,avg_rating,pages,genres,url
0,20412869.0,Study Guide: The Perks of Being a Wallflower b...,3.98,38,Fiction,https://www.goodreads.com/book/show/20412869.0
1,49676332.0,Study Guide: Nineteen Minutes by Jodi Picoult,4.00,80,,https://www.goodreads.com/book/show/49676332.0
2,6149.0,Beloved,3.98,325,"Magical Realism, Classics, School",https://www.goodreads.com/book/show/6149.0
3,12437190.0,Crank by Ellen Hopkins l Summary & Study Guide,4.60,N/A,,https://www.goodreads.com/book/show/12437190.0
4,19365546.0,Summary & Study Guide Fallout by Ellen Hopkins,4.67,31,,https://www.goodreads.com/book/show/19365546.0
